***Protein Protein Interaction Project***

*Authors*: Jordan Harrison, Haoyue Hu

In [1]:
# Import necessary Libraries

import pandas as pd

In [ ]:
# Load the dataset
loaded = False


"""
have this in just as a way to check if the files are in the right place. the github is structured to have two data
folders (don't ask me why I don't know), but the when we submit the files could be in the same dir
"""
try:
    int_data = pd.read_csv("Data/Data/Supp-AB.tsv", sep='\t')
    seq_data = pd.read_csv("Data/Data/SEQ-Supp-ABCD.tsv", sep='\t', header = None, names = ['id', 'seq'])
    seq_map = dict(zip(seq_data.id, seq_data.seq))
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print("Checking local directory for files...")
try:
    int_data = pd.read_csv("Supp-AB.tsv", sep='\t')
    seq_data = pd.read_csv("SEQ-Supp-ABCD.tsv", sep='\t', header = None)
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print("Files not found in local directory")

# Check everything loaded correctly
pair_ids = set(int_data.v1) | set(int_data.v2)
print(f"Number of interacting protein pairs: {len(int_data)}")
print(f"Number of protein sequences: {len(seq_data)}")
print(f"Number of unique protein pairs: {len(pair_ids)}")

# Check data set
print(int_data.head())
#int_data.info()
print(seq_data.head())
#seq_data.info()

Data loaded successfully.
Files not found in local directory
Number of interacting protein pairs: 73108
Number of protein sequences: 10370
Number of unique protein pairs: 10364
set()
               v1              v2  label
0     NP_663777.1     NP_001233.1      1
1     NP_055436.2     NP_001420.2      1
2     NP_003630.1  NP_001073594.1      1
3  NP_001001998.1     NP_067000.1      1
4     NP_001888.2     NP_001782.1      1
       id                                                seq
0  A0AVK6  MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...
1  A2A3N6  MASEVPYASGMPIKKIGHRSVDSSGGTTSSALKGAIQLGITHTVGS...
2  A2A3N6  MASEVPYASGMPIKKIGHRSVDSSGGTTSSALKGAIQLGITHTVGS...
3  A6H8Y1  MFRRARLSVKPNVRPGVGARGSTASNPQRGRESPRPPEPATDSASK...
4  A6H8Y1  MFRRARLSVKPNVRPGVGARGSTASNPQRGRESPRPPEPATDSASK...


In [16]:
# create new dataframe with the protein sequences and their interactions
df = int_data.copy()
df['protein_A_seq'] = df.v1.map(seq_map)
df['protein_B_seq'] = df.v2.map(seq_map)

"""
To avoid redunancy, the pair AB and BA are the same thing so we need to check for any duplicates
"""
# wizardry here, creates columns 'a' and 'b' then uses the min and max (which works alphabetically) to order the columns
# if you get ab and the ba this function puts ab into both of the new columns B)
df['a'] = df[['v1', 'v2']].min(axis=1)
df['b'] = df[['v1', 'v2']].max(axis=1)
df = df.drop_duplicates(subset=['a','b','label']).reset_index(drop=True)

# clean it up, apply good labels, see the balance
df = df.rename(columns={'v1':'id_A','v2':'id_B'})[['id_A','id_B','protein_A_seq','protein_B_seq','label']]
print(df.label.value_counts(), df.shape)



label
1    36629
0    36479
Name: count, dtype: int64 (73108, 5)
